# Aula 13: construindo o primeiro agente

Hoje vocês vão montar, com as próprias mãos, exatamente o que foi explicado
nos slides: um agente com uma ferramenta, rodando o loop pensa, age, observa,
responde, sobre o mesmo caso de fraude em cartão que vocês já conhecem desde a
Aula 2.

A ferramenta não é simulada. É um **servidor MCP de verdade**, o mesmo
protocolo que apareceu no Bloco 02 (história) e no Bloco 05 (controle) da
aula de hoje.

**Antes de começar:** você precisa de uma chave gratuita da NVIDIA. Veja o
Passo 2 abaixo.

## Passo 0 · Instalar as bibliotecas

Duas bibliotecas novas hoje: `fastmcp`, para construir o servidor MCP, e
`openai`, cujo cliente também funciona com a NVIDIA, porque a API da NVIDIA
segue o mesmo formato da OpenAI.

In [ ]:
!pip install fastmcp openai -q

print("Bibliotecas instaladas.")

## Passo 1 · Baixar os dados

As mesmas tabelas de sempre: transações e lojistas, do caso de fraude em
cartão.

In [ ]:
import gdown
import os

files = {
    'transacoes.csv': '1OR3lgLjqVQZoajau2wrSExWPYc8xUsFS',
    'lojistas.csv': '1vydqkbd2fONvNAfNKon9izr0pqaGBqhy',
}

os.makedirs('csv', exist_ok=True)
for filename, file_id in files.items():
    gdown.download(f'https://drive.google.com/uc?id={file_id}', f'csv/{filename}', quiet=True)
    print(f"Baixado: {filename}")

PASTA_DADOS = "/content/csv"

## Passo 2 · A chave da NVIDIA

O "cérebro" do agente hoje é um modelo hospedado gratuitamente pela NVIDIA,
pelo mesmo tipo de API que a OpenAI usa (por isso o cliente Python é o mesmo).

1. Acesse **build.nvidia.com** e crie uma conta gratuita.
2. Vá em **Settings → API Keys** e clique em **Generate Key**.
3. Copie a chave (começa com `nvapi-`) e cole abaixo.

A chave só aparece uma vez na tela da NVIDIA. Se perder, gere outra.

In [ ]:
import getpass

# Cola a chave quando pedir. Ela fica só na memória desta sessão, não é salva
# em lugar nenhum do notebook.
NVIDIA_API_KEY = getpass.getpass("Cole sua chave da NVIDIA (nvapi-...): ")

## Passo 3 · O servidor MCP, com uma ferramenta só

Uma ferramenta genérica e simples: dado o número de um lojista, devolve
quantas transações ele teve e qual a taxa de contestação dele.

Reparem na `docstring` da função: é ela que o agente vai ler para decidir
quando essa ferramenta é útil. Isso é o mesmo "contrato" descrito no Bloco 03
de hoje: o modelo só enxerga o nome, a descrição e os parâmetros, nunca o
código por dentro.

In [ ]:
import pandas as pd
from fastmcp import FastMCP

transacoes = pd.read_csv(f"{PASTA_DADOS}/transacoes.csv")
lojistas = pd.read_csv(f"{PASTA_DADOS}/lojistas.csv")

mcp = FastMCP("Ferramentas da Aula 13")


@mcp.tool
def consultar_lojista(id_lojista: int) -> dict:
    """Consulta o total de transacoes e a taxa de contestacao de um lojista,
    pelo id_lojista. Use esta ferramenta sempre que precisar saber se um
    lojista especifico tem um padrao de contestacao alto ou baixo."""
    dados = transacoes[transacoes["id_lojista"] == id_lojista]
    if dados.empty:
        return {"erro": f"nenhum lojista encontrado com id {id_lojista}"}
    total = len(dados)
    contestadas = (dados["status"] == "contestada").sum()
    taxa = round(100 * contestadas / total, 2)
    nome_row = lojistas[lojistas["id_lojista"] == id_lojista]
    nome = nome_row["nome"].iloc[0] if not nome_row.empty else "desconhecido"
    return {
        "id_lojista": id_lojista,
        "nome_lojista": nome,
        "total_transacoes": total,
        "transacoes_contestadas": int(contestadas),
        "taxa_contestacao_pct": taxa,
    }


print("Servidor MCP criado, com a ferramenta consultar_lojista.")

## Passo 4 · Testando a ferramenta, sem o modelo ainda

Antes de conectar o agente, vale confirmar que a ferramenta em si funciona,
chamando ela através de um cliente MCP diretamente. Isso conecta ao servidor
**dentro do mesmo notebook**, sem precisar de um processo separado nem de
rede, o jeito mais simples de rodar um servidor MCP para aprender.

In [ ]:
from fastmcp import Client

cliente_mcp = Client(mcp)

async with cliente_mcp:
    ferramentas = await cliente_mcp.list_tools()
    print("Ferramentas que o agente vai enxergar:")
    for f in ferramentas:
        print(f"- {f.name}: {f.description.strip().splitlines()[0]}")

    print()
    resultado = await cliente_mcp.call_tool("consultar_lojista", {"id_lojista": 141})
    print("Resultado de teste, lojista 141:")
    print(resultado.data)

## Passo 5 · Conectando o cérebro à ferramenta

O modelo de linguagem não fala "MCP" nativamente, ele fala o formato de
ferramenta da própria API. Esta célula traduz a lista de ferramentas do
servidor MCP para esse formato, uma vez só, para reaproveitar depois.

In [ ]:
from openai import OpenAI

client_llm = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=NVIDIA_API_KEY,
)

MODELO = "nvidia/nemotron-3.5-lightning-30b-a3b"


async def formato_das_ferramentas():
    async with Client(mcp) as c:
        ferramentas = await c.list_tools()
    return [
        {
            "type": "function",
            "function": {
                "name": f.name,
                "description": f.description,
                "parameters": f.input_schema,
            },
        }
        for f in ferramentas
    ]


tools_formato_openai = await formato_das_ferramentas()
print("Ferramenta traduzida para o formato da API:")
print(tools_formato_openai)

## Passo 6 · O loop do agente

Isto aqui é o loop ReAct do Bloco 03, em código de verdade: pensa, age,
observa, e repete até não precisar mais de nenhuma ferramenta.

Cada `print` abaixo corresponde a uma das quatro palavras que apareceram nos
slides.

In [ ]:
import json


async def perguntar_ao_agente(pergunta, max_passos=5):
    # Uma instrução de sistema clara ajuda o modelo a responder em português
    # e a usar a ferramenta quando for o caso, seguindo o formato ReAct.
    mensagens = [
        {
            "role": "system",
            "content": "Você é um assistente útil com acesso a ferramentas. Responda em português. Use as ferramentas disponibilizadas quando necessário.",
        },
        {"role": "user", "content": pergunta},
    ]

    async with Client(mcp) as cliente_mcp:
        for passo in range(max_passos):
            print(f"--- passo {passo + 1} ---")
            print("PENSA: mandando a pergunta e o histórico para o modelo")

            # temperature=0.0 deixa o modelo mais previsível: menos chance de
            # alucinar ou pular a ferramenta à toa.
            resposta = client_llm.chat.completions.create(
                model=MODELO,
                messages=mensagens,
                tools=tools_formato_openai,
                temperature=0.0,
            )
            msg = resposta.choices[0].message

            if not msg.tool_calls:
                print("RESPONDE:", msg.content)
                return msg.content

            # guarda a fala do modelo, incluindo o pedido de ferramenta, no historico
            mensagens.append({
                "role": "assistant",
                "content": msg.content,
                "tool_calls": [tc.model_dump() for tc in msg.tool_calls],
            })

            for chamada in msg.tool_calls:
                nome = chamada.function.name
                argumentos = json.loads(chamada.function.arguments)
                print(f"AGE: chamando {nome} com {argumentos}")

                resultado = await cliente_mcp.call_tool(nome, argumentos)
                print(f"OBSERVA: {resultado.data}")

                mensagens.append({
                    "role": "tool",
                    "tool_call_id": chamada.id,
                    "content": json.dumps(resultado.data, ensure_ascii=False),
                })

    print("O agente não terminou dentro do limite de passos.")
    return None


print("Função perguntar_ao_agente pronta, com prompt de sistema e temperature=0.0.")

## Passo 7 · Testando o agente de verdade

Agora sim. Pergunte alguma coisa que só dá para responder consultando a
ferramenta.

In [ ]:
resposta = await perguntar_ao_agente("O lojista 141 tem uma taxa de contestação alta? Qual é o número exato?")

Teste também uma pergunta que **não precisa** da ferramenta, para ver o
agente decidir sozinho que não é necessário agir desta vez.

In [ ]:
resposta = await perguntar_ao_agente("Explique em uma frase o que é uma taxa de contestação.")

E uma pergunta que faz sentido testar em grupo: escolham um `id_lojista`
diferente e perguntem alguma coisa sobre ele.

In [ ]:
resposta = await perguntar_ao_agente("Consulte o lojista 7 e me diga se ele parece confiável.")

## Pronto

Vocês acabaram de montar, com as próprias mãos, os quatro conceitos do dia:
um agente (Bloco 03), com uma ferramenta real servida por MCP (Bloco 02 e
05), decidindo sozinho quando agir (Bloco 04), sobre o caso de fraude que já
conhecem.

**Pergunta de reflexão:** na pergunta que não precisava da ferramenta, o
agente chamou `consultar_lojista` mesmo assim? Se chamou, por que vocês acham
que isso aconteceu, e o que isso tem a ver com o bloco de controle de hoje?